# RBM Training with Cross-Validation

This notebook trains a Restricted Boltzmann Machine on MNIST digits
**0 / 1 / 2** with the following hyperparameters:

| Parameter | Value |
|-----------|-------|
| Hidden units *L* | 16 |
| CD steps *Nt* | 10 |
| L1 penalty *γ* | 0.0001 |
| Optimizer | RMSprop |
| Learning rate | 0.001 |
| Spins / Potts | False / False |

The notebook includes:

1. **Train / Validation monitoring** — the dataset is split 80 / 20 and
   metrics (reconstruction error, pseudo-likelihood, free-energy gap)
   are computed on both sets at regular intervals to detect overfitting
   (Hinton, Section 6).
2. **K-Fold cross-validation** — 5-fold CV to estimate the statistical
   uncertainty of the final metrics.

> Run cells **top to bottom**. The notebook is self-contained.

## 0. Setup and Imports

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import KFold
from joblib import Parallel, delayed

from rbm import RBM
from rbm_plots import plot_filters, plot_metrics

plt.rcParams.update({"font.size": 12})

## 1. Data Loading and Preprocessing

In [ ]:
X_original, Y_original = fetch_openml("mnist_784", version=1,
                                      return_X_y=True, as_frame=False)

X_original = np.array(X_original, dtype=np.float32)
Y_original = np.array(Y_original, dtype=int)

mask = (Y_original == 0) | (Y_original == 1) | (Y_original == 2)
X_all = X_original[mask]
Y_all = Y_original[mask]

def preprocess_mnist(images, seed=None):
    rng = np.random.RandomState(seed)
    images = images / 255.0
    images = images + rng.normal(0, 0.01, images.shape)
    return np.clip(images, 0, 1)

X_all = preprocess_mnist(X_all, seed=42)
D = X_all.shape[1]  # 784

print(f"Dataset : {X_all.shape}")
print(f"Range   : [{X_all.min():.3f}, {X_all.max():.3f}]")
print(f"Mean    : {X_all.mean():.3f}")

## 2. Hyperparameters

In [ ]:
# --- Hyperparameters --------------------------------
HPARAMS = dict(
    L          = 16,
    Nt         = 10,
    gamma      = 0.0001,
    optimizer  = "rmsprop",
    lr         = 0.001,
    spins      = False,
    potts      = False,
)

# --- Training configuration ----------------------------------------
NEPOCH       = 150
BATCH_SIZE   = 100
W_INIT_SCALE = 0.01     # Hinton-recommended
METRICS_EVERY = 5       # compute metrics every N epochs
SEED         = 42

print("Model hyperparameters:")
for k, v in HPARAMS.items():
    print(f"  {k:12s} = {v}")
print(f"\nTraining: {NEPOCH} epochs, batch {BATCH_SIZE}, "
      f"w_init {W_INIT_SCALE}, metrics every {METRICS_EVERY} epochs")

## 3. Data Splits

We use a **three-way split**:

| Split | Size | Purpose |
|-------|------|---------|
| `X_test` | 15 % | **held-out test set** — never used during training or CV |
| `X_cv`   | 85 % | k-fold cross-validation pool |
| └ `X_train` / `X_val` | 80 / 20 % of X_cv | single-model training with monitoring |

This guarantees that the final reconstruction visualisation and metrics
are computed on samples the model has genuinely never seen.

In [ ]:
rng = np.random.RandomState(SEED)
idx = rng.permutation(X_all.shape[0])

n_test = int(0.15 * len(idx))
idx_test = idx[:n_test]
idx_cv   = idx[n_test:]

X_test = X_all[idx_test]   # held-out — never touched until section 9
X_cv   = X_all[idx_cv]     # used for CV folds

# 80/20 of X_cv for the monitored single-model training (sections 4-7)
n_val    = int(0.20 * len(idx_cv))
idx_val  = idx_cv[:n_val]
idx_train = idx_cv[n_val:]
X_train  = X_all[idx_train]
X_val    = X_all[idx_val]

print(f"Test set          : {X_test.shape[0]:6d} samples  (held-out)")
print(f"CV pool           : {X_cv.shape[0]:6d} samples")
print(f"  Single-model train : {X_train.shape[0]:6d} samples")
print(f"  Single-model val   : {X_val.shape[0]:6d} samples")

## 4. Training with Train / Val Monitoring

We use a manual epoch loop (calling `rbm.update()` on each mini-batch)
so that we can compute metrics on **both** training and validation sets
at regular intervals. This gives proper train-vs-val learning curves.

In [ ]:
# --- Create the model ----------------------------------------------
model = RBM(
    n_v=D,
    n_h=HPARAMS["L"],
    optimizer=HPARAMS["optimizer"],
    lr=HPARAMS["lr"],
    lr_schedule="constant",
    gamma=HPARAMS["gamma"],
    weight_decay=0.0,
    spins=HPARAMS["spins"],
    potts=HPARAMS["potts"],
    w_init_scale=W_INIT_SCALE,
    seed=SEED,
)

# Hinton visible-bias initialization
p = np.clip(X_train.mean(axis=0), 0.001, 0.999)
model.v_bias = np.log(p / (1 - p))

# --- Tracking containers ------------------------------------------
history = {
    "epoch":    [],
    "train_re": [], "val_re": [],
    "train_pl": [], "val_pl": [],
    "train_fe": [], "val_fe": [],
    "grad_rms": [], "sparsity": [], "update_ratio": [],
}

n_train   = X_train.shape[0]
n_batches = n_train // BATCH_SIZE

# Subset for fast metric computation
n_met = min(1000, n_train)
X_met_train = X_train[:n_met]
X_met_val   = X_val[:min(n_met, X_val.shape[0])]

print(f"Training: {NEPOCH} epochs, {n_batches} batches/epoch, "
      f"CD-{HPARAMS['Nt']}\n")

# --- Training loop -------------------------------------------------
for epoch in range(1, NEPOCH + 1):
    perm = rng.permutation(n_train)
    epoch_grms = []
    epoch_sp   = []
    epoch_ur   = []

    lr  = model.lr
    mom = model._get_momentum(epoch)

    for b in range(n_batches):
        batch = X_train[perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]]
        stats = model.update(batch, n_cd=HPARAMS["Nt"],
                             persistent=False, lr=lr, momentum=mom)
        epoch_grms.append(np.std(stats["dW"]))
        epoch_sp.append(stats["sparsity"])
        epoch_ur.append(stats["update_ratio"])

    # --- Compute metrics on train AND val ---
    if epoch % METRICS_EVERY == 0 or epoch == 1:
        train_re = model.reconstruction_error(X_met_train)
        val_re   = model.reconstruction_error(X_met_val)
        train_pl = model.pseudo_likelihood(X_met_train)
        val_pl   = model.pseudo_likelihood(X_met_val)
        train_fe = model.free_energy(X_met_train)
        val_fe   = model.free_energy(X_met_val)

        history["epoch"].append(epoch)
        history["train_re"].append(train_re)
        history["val_re"].append(val_re)
        history["train_pl"].append(train_pl)
        history["val_pl"].append(val_pl)
        history["train_fe"].append(train_fe)
        history["val_fe"].append(val_fe)
        history["grad_rms"].append(np.mean(epoch_grms))
        history["sparsity"].append(np.mean(epoch_sp))
        history["update_ratio"].append(np.mean(epoch_ur))

        if epoch % 25 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{NEPOCH}  "
                  f"RE(t/v)={train_re:.4f}/{val_re:.4f}  "
                  f"PL(t/v)={train_pl:.1f}/{val_pl:.1f}  "
                  f"FE(t/v)={train_fe:.1f}/{val_fe:.1f}  "
                  f"gradRMS={np.mean(epoch_grms):.5f}")

print("\nTraining complete.")

## 5. Train / Validation Learning Curves

Comparing training and validation metrics detects overfitting: when the
validation curve stalls or worsens while the training curve keeps
improving, the model is memorizing rather than generalizing.

In [ ]:
ep = history["epoch"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

# Row 1: train vs val
ax = axes[0, 0]
ax.plot(ep, history["train_re"], "o-", ms=3, label="Train")
ax.plot(ep, history["val_re"],   "s--", ms=3, label="Validation")
ax.set(xlabel="Epoch", ylabel="MSE", title="Reconstruction Error")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(ep, history["train_pl"], "o-", ms=3, label="Train")
ax.plot(ep, history["val_pl"],   "s--", ms=3, label="Validation")
ax.set(xlabel="Epoch", ylabel="log PL", title="Pseudo-Likelihood")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 2]
ax.plot(ep, history["train_fe"], "o-", ms=3, label="Train")
ax.plot(ep, history["val_fe"],   "s--", ms=3, label="Validation")
ax.set(xlabel="Epoch", ylabel="F(v)", title="Free Energy")
ax.legend(); ax.grid(alpha=0.3)

# Row 2: diagnostics
ax = axes[1, 0]
ax.plot(ep, history["grad_rms"], "o-", ms=3, color="#228833")
ax.set(xlabel="Epoch", ylabel="RMS", title="Gradient RMS")
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(ep, history["sparsity"], "o-", ms=3, color="#CC4422")
ax.set(xlabel="Epoch", ylabel="Mean h", title="Hidden Sparsity")
ax.grid(alpha=0.3)

ax = axes[1, 2]
ax.plot(ep, history["update_ratio"], "o-", ms=3, color="#884488")
ax.set(xlabel="Epoch", ylabel="||step|| / ||W||",
       title="Update / Weight Ratio")
ax.grid(alpha=0.3)

fig.suptitle(f"Train / Validation Monitoring  "
             f"(L={HPARAMS['L']}, Nt={HPARAMS['Nt']}, "
             f"lr={HPARAMS['lr']}, gamma={HPARAMS['gamma']})",
             fontsize=13)
plt.show()

## 6. Learned Filters

In [ ]:
plot_filters(model.W, side=28,
             title=f"Learned filters — L={HPARAMS['L']}, Nt={HPARAMS['Nt']}")

## 7. Visual Reconstruction (Validation Set)

In [ ]:
np.random.seed(0)
idx = np.random.choice(X_val.shape[0], 8, replace=False)
samples = X_val[idx]
h_prob, _ = model.sample_h(samples)
recon, _  = model.sample_v(h_prob)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(samples[i].reshape(28, 28), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].reshape(28, 28), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Recon", fontsize=11)
fig.suptitle("Validation-set reconstruction", fontsize=13)
plt.tight_layout()
plt.show()

## 8. K-Fold Cross-Validation (K = 5) — Parallelized

We run 5-fold CV on **`X_cv`** (the 85 % pool), keeping `X_test` fully
held out. Each fold trains on 4/5 of `X_cv` and validates on the
remaining 1/5, giving **mean ± std** estimates of the final metrics
(assignment Note 1).

The 5 folds run **in parallel** via `joblib`. On an Apple Silicon
M1 Pro (8–10 cores) this gives a near-linear speedup: wall-clock time
drops from ~5× to ~1× the single-fold cost.

> **Thread management:** each worker inherits `VECLIB_MAXIMUM_THREADS = 2`
> to limit per-process Accelerate BLAS threads and avoid oversubscription.

In [ ]:
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"]   = "2"
os.environ["MKL_NUM_THREADS"]        = "2"

K  = 5
kf = KFold(n_splits=K, shuffle=True, random_state=SEED)

def train_one_fold(fold_idx, train_idx, val_idx):
    """Train one CV fold on X_cv; return metrics."""
    Xtr = X_cv[train_idx]
    Xvl = X_cv[val_idx]

    m = RBM(
        n_v=D,
        n_h=HPARAMS["L"],
        optimizer=HPARAMS["optimizer"],
        lr=HPARAMS["lr"],
        lr_schedule="constant",
        gamma=HPARAMS["gamma"],
        weight_decay=0.0,
        spins=HPARAMS["spins"],
        potts=HPARAMS["potts"],
        w_init_scale=W_INIT_SCALE,
        seed=SEED + fold_idx,
    )
    m.train(
        Xtr,
        Nepoch=NEPOCH,
        Nmini=len(Xtr) // BATCH_SIZE,
        N_ini=BATCH_SIZE,
        N_fin=BATCH_SIZE,
        Nt=HPARAMS["Nt"],
        persistent=False,
        hinton_init=True,
        metrics_every=0,
        history_every=0,
        verbose=False,
    )

    Xtr_sub = Xtr[:2000]
    Xvl_sub = Xvl[:2000]

    return {
        "fold":     fold_idx,
        "re_train": m.reconstruction_error(Xtr_sub),
        "re_val":   m.reconstruction_error(Xvl_sub),
        "pl_train": m.pseudo_likelihood(Xtr_sub),
        "pl_val":   m.pseudo_likelihood(Xvl_sub),
        "fe_train": m.free_energy(Xtr_sub),
        "fe_val":   m.free_energy(Xvl_sub),
    }

folds = list(kf.split(X_cv))

print(f"{K}-Fold CV on X_cv ({X_cv.shape[0]} samples), "
      f"{NEPOCH} epochs/fold, {K} parallel jobs\n")

t0 = time.perf_counter()
results_list = Parallel(n_jobs=K, verbose=10)(
    delayed(train_one_fold)(fold + 1, tr, vl)
    for fold, (tr, vl) in enumerate(folds)
)
elapsed = time.perf_counter() - t0
print(f"\nAll {K} folds done in {elapsed:.1f}s  ({elapsed/K:.1f}s/fold effective)\n")

cv_results = {"re_train": [], "re_val": [],
              "pl_train": [], "pl_val": [],
              "fe_train": [], "fe_val": []}

for r in sorted(results_list, key=lambda x: x["fold"]):
    for key in cv_results:
        cv_results[key].append(r[key])
    print(f"  Fold {r['fold']}/{K}  "
          f"RE(t/v)={r['re_train']:.4f}/{r['re_val']:.4f}  "
          f"PL(t/v)={r['pl_train']:.1f}/{r['pl_val']:.1f}")

## 9. Cross-Validation Results

In [ ]:
# --- CV summary table ---
print(f"{'Metric':<25s} {'Train (CV)':>20s}   {'Val (CV)':>20s}")
print("-" * 70)
for label, key_t, key_v in [
    ("Reconstruction Error", "re_train", "re_val"),
    ("Pseudo-Likelihood",    "pl_train", "pl_val"),
    ("Free Energy",          "fe_train", "fe_val"),
]:
    mt = np.mean(cv_results[key_t]);  st = np.std(cv_results[key_t])
    mv = np.mean(cv_results[key_v]);  sv = np.std(cv_results[key_v])
    print(f"{label:<25s} {mt:8.4f} +/- {st:.4f}   {mv:8.4f} +/- {sv:.4f}")

# --- Bar chart ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, (title, key_t, key_v) in zip(axes, [
    ("Reconstruction Error", "re_train", "re_val"),
    ("Pseudo-Likelihood",    "pl_train", "pl_val"),
    ("Free Energy",          "fe_train", "fe_val"),
]):
    means = [np.mean(cv_results[key_t]), np.mean(cv_results[key_v])]
    stds  = [np.std(cv_results[key_t]),  np.std(cv_results[key_v])]
    ax.bar(["Train (CV)", "Val (CV)"], means, yerr=stds,
           color=["#2266CC", "#CC4422"], alpha=0.8,
           capsize=6, edgecolor="black", linewidth=0.5)
    ax.set(title=title, ylabel=title)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(f"{K}-Fold CV Results  (L={HPARAMS['L']}, Nt={HPARAMS['Nt']}, "
             f"lr={HPARAMS['lr']}, gamma={HPARAMS['gamma']})", fontsize=13)
plt.show()

re_gap = np.mean(cv_results["re_val"]) - np.mean(cv_results["re_train"])
print(f"\nOverfitting diagnostic:  RE gap (val - train) = {re_gap:.5f}")
if   re_gap < 0.005: print("  -> Minimal overfitting.")
elif re_gap < 0.020: print("  -> Moderate overfitting.")
else:                print("  -> Significant overfitting.")

## 10. Final Evaluation on the Held-Out Test Set

`X_test` was never used during training or cross-validation.
We now evaluate the **main model** (trained in section 4) on it
to get an unbiased estimate of generalisation performance.

In [ ]:
# --- Quantitative metrics on X_test ---
test_re = model.reconstruction_error(X_test)
test_pl = model.pseudo_likelihood(X_test)
test_fe = model.free_energy(X_test)

cv_re_mean = np.mean(cv_results["re_val"])
cv_re_std  = np.std(cv_results["re_val"])

print(f"Test-set metrics  (N = {X_test.shape[0]})")
print(f"  Reconstruction Error : {test_re:.4f}")
print(f"  Pseudo-Likelihood    : {test_pl:.2f}")
print(f"  Free Energy          : {test_fe:.2f}")
print()
print(f"CV val RE (mean +/- std) : {cv_re_mean:.4f} +/- {cv_re_std:.4f}")
print(f"Test RE                  : {test_re:.4f}")
diff = test_re - cv_re_mean
print(f"Difference               : {diff:+.4f}  "
      f"({'within' if abs(diff) <= cv_re_std else 'outside'} 1 CV std)")

# --- Visual reconstruction ---
np.random.seed(7)
idx = np.random.choice(X_test.shape[0], 8, replace=False)
samples = X_test[idx]
h_prob, _ = model.sample_h(samples)
recon, _  = model.sample_v(h_prob)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(samples[i].reshape(28, 28), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].reshape(28, 28),   cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Recon",    fontsize=11)
fig.suptitle(f"Test-set reconstruction  (RE = {test_re:.4f})", fontsize=13)
plt.tight_layout()
plt.show()